File Name: ggm_assessment.ipynb

Author:
    Debora Bisoglio
    Carmine Frascella
    B-Open Solutions

Date: August 2025

Description:
------------
This notebook computes the algorithm stability for the GGM product.

Usage:
------
To use this script, define the following parameter:
    
    - sigma_input


In [43]:
#!/usr/bin/python3 
import numpy as np

In [44]:
#set the 4 sigmas for the satellite you want to analyse (asc-VV, asc-VH, desc-VV, desc_VH)
sigma_input = [0.001, 0.04, 0.001, 0.04] #S1 100m precision
#sigma_input = [0.0005, 0.02, 0.0005, 0.02] #S1 200m precision
#sigma_input = [0.002, 0.03, 0.002, 0.03] #rosel 100m precision
#sigma_input = [0.001, 0.015, 0.001, 0.015] #rosel 200m precision


In [45]:
def ortogonale(vec):
    matr = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 0]])
    return np.dot(matr, vec)

In [46]:
def get_along_track_vector(vec):
    """
    Returns a unit vector orthogonal to `vec` (3D),
    with the constraint that the resulting vector lies in the XY plane (i.e., z = 0).
    This is done by projecting `vec` onto XY and rotating 90° in-plane.
    """
    v_xy = np.array([vec[0], vec[1]])
    if np.linalg.norm(v_xy) < 1e-8:
        raise ValueError("Cannot compute orthogonal vector: XY projection is near zero.")
    ortho_xy = np.array([-v_xy[1], v_xy[0], 0])
    ortho_unit = ortho_xy / np.linalg.norm(ortho_xy)
    return ortho_unit

def get_matrix():
    """
    Generates a matrix L of 4 unit LOS vectors:
    2 real (ASC and DESC), and 2 along-track vectors in the XY plane.
    Adds random variation to simulate realistic LOS differences.
    """
    asc = np.array([
        -0.1  - 0.02 * np.random.rand(), 
        -0.65 - 0.07 * np.random.rand(), 
        0.7   + 0.18 * np.random.rand()
    ])
    desc = np.array([
        -0.1  - 0.02 * np.random.rand(), 
        0.65  + 0.07 * np.random.rand(), 
        0.7   + 0.18 * np.random.rand()
    ])
    asc /= np.linalg.norm(asc)
    desc /= np.linalg.norm(desc)

    return np.array([
                asc,
                get_along_track_vector(asc),
                desc,
                get_along_track_vector(desc)
            ])

In [47]:
A = get_matrix()
A[0, 0] = 0
A[2, 0] = 0
A[0, :] /= np.linalg.norm(A[0, :])
A[2, :] /= np.linalg.norm(A[2, :])

In [48]:
d = np.array([np.random.normal(scale=0.5, size=100000), np.random.normal(scale=0.4, size=100000), np.random.normal(scale=0.5, size=100000), np.random.normal(scale=0.4, size=100000)])
V = np.array((0.005, 0.4, 0.005, 0.4))[..., None]
A_prime = A / (V ** 2)
d_prime = d / (V ** 2)

In [49]:
L_prime = np.linalg.inv(A_prime.transpose() @ A_prime) @ A_prime.transpose()
L = np.linalg.inv(A.transpose() @ A) @ A.transpose()
Y = L @ d
Y_prime = L_prime @ d_prime

In [50]:
print(np.std(Y, axis=1))
print(np.std(Y_prime, axis=1))

[0.28589992 0.51839092 0.47089766]
[0.28589968 0.53684232 0.4708988 ]


In [51]:
def find_sigma(L, sigma):
    B = np.linalg.inv(L.transpose()@L)@L.transpose()
    return np.sqrt(B**2@sigma**2)

In [52]:
def simulation_satellite(sigma_input):
    L = get_matrix()
    sigma = find_sigma(L, np.array(sigma_input))
    return sigma

In [53]:
sigmas_sat = []

for i in range(100000):
    sigmas_sat.append(simulation_satellite(sigma_input))
    
sigmas_sat= np.array(sigmas_sat).transpose()
max_sigmas = np.max(sigmas_sat, axis = 1)